# 10c -- MTMC Stages 4-5: Association + Evaluation

**Prerequisite**: attach **10b's output** as a data source:
`Add Data -> Kernel Output -> search "mtmc-10b-stage-3-faiss-indexing" -> add`

**This is the iteration loop** -- edit the tuning params cell and re-run in ~6 min.
No GPU needed, no data download, no model inference.

| Stage | What | Time |
|---|---|---|
| 4 | Cross-camera association (AQE + Louvain graph clustering) | ~5 min |
| 5 | Evaluation: IDF1, MOTA, HOTA | ~1 min |

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile
from pathlib import Path
import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT  = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"\n\u2713 Repo ready at {PROJECT}")

In [ ]:
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

try:
    import faiss; print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try: pip("faiss-gpu")
    except Exception: pip("faiss-cpu")

pip("motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click",
    "numpy", "scipy", "pandas", "scikit-learn")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
                      cwd=str(PROJECT))
print("\n\u2713 Dependencies installed")

In [ ]:
FAILED = []
_checks = [
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("loguru", "loguru"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("sklearn", "sklearn"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
]
for label, mod in _checks:
    try:
        __import__(mod)
        print(f"  \u2713 {label}")
    except ImportError as e:
        print(f"  \u2717 {label}: {e}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED} -- fix pip installs above")
print("\n\u2713 All required modules importable")

## 2. Load Checkpoint from 10b

Finds `checkpoint.tar.gz` in `/kaggle/input/mtmc-10b-stage-3-faiss-indexing/` and extracts it.

In [ ]:
PREV_SLUG = "mtmc-10b-stage-3-faiss-indexing"
PREV_INPUT = Path("/kaggle/input") / PREV_SLUG

if not PREV_INPUT.exists():
    for p in Path("/kaggle/input").iterdir():
        if PREV_SLUG in p.name or "stage3" in p.name or "10b" in p.name:
            PREV_INPUT = p; break

cp = PREV_INPUT / "checkpoint.tar.gz"
assert cp.exists(), f"checkpoint.tar.gz not found at {cp}"

EXTRACT_DIR = Path("/tmp/pipeline_run")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {cp.stat().st_size/1024**2:.1f} MB ...")
with tarfile.open(str(cp), "r:gz") as tar:
    tar.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json") as f:
    meta = json.load(f)
RUN_NAME = meta["run_name"]
DATA_OUT = EXTRACT_DIR
RUN_DIR  = EXTRACT_DIR / RUN_NAME
GT_DIR   = str(EXTRACT_DIR / "gt_annotations")
print(f"\u2713 Checkpoint extracted -- run: {RUN_NAME}")
for s in ["stage1", "stage2", "stage3"]:
    d = RUN_DIR / s
    if d.exists(): print(f"  {s}: {len(list(d.rglob('*')))} files")
print(f"  GT dir: {GT_DIR}")

## 3. Tuning Parameters

**Edit these values** then re-run the cells below (~6 min). No need to re-run 10a or 10b.

In [ ]:
# ---- Stage 4: Cross-camera association -------------------------------------
# AQE: k nearest neighbours for query expansion (higher = smoother features)
AQE_K             = 5

# Minimum cosine similarity to form an edge in the Louvain graph
SIM_THRESH        = 0.35

# Louvain resolution (higher = more, smaller clusters)
LOUVAIN_RES       = 0.8

# Weight of appearance vs. spatio-temporal score (0.0=ST only, 1.0=appear only)
APPEARANCE_WEIGHT = 0.6

# ---- Stage 5: Evaluation ----------------------------------------------------
# CityFlowV2 GT includes BOTH multi-cam (81 in S01, 130 in S02) AND
# single-cam (14 in S01, 15 in S02) vehicles — total 240 annotated vehicles.
# Setting True filters single-cam predictions, dropping 29 GT vehicles → lower
# IDF1.  Keep False for AIC-protocol comparison against published SOTA (84.1%).
MTMC_ONLY = False

print("Stage 4 params:")
print(f"  aqe_k={AQE_K}  sim_thresh={SIM_THRESH}  louvain_res={LOUVAIN_RES}  appearance_weight={APPEARANCE_WEIGHT}")
print(f"Stage 5: mtmc_only_submission={MTMC_ONLY}")

## 4. Run Stages 4-5

In [ ]:
os.chdir(str(PROJECT))
cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/cityflowv2.yaml",
    "--stages", "4,5",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
    "--override", f"stage4.association.query_expansion.k={AQE_K}",
    "--override", f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
    "--override", f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
    "--override", f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
    "--override", f"stage5.ground_truth_dir={GT_DIR}",
    "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
]
print("CMD:", " ".join(str(c) for c in cmd))
print("=" * 70)
t0 = time.time()
r = subprocess.run(cmd, cwd=str(PROJECT))
print("=" * 70)
elapsed = time.time() - t0
if r.returncode != 0:
    print(f"\u2717 FAILED after {elapsed/60:.1f} min"); sys.exit(r.returncode)
print(f"\u2713 Stages 4-5 done in {elapsed/60:.1f} min")

## 5. Results

In [ ]:
stage5_dir = RUN_DIR / "stage5"

def _pct(v):
    return f"{v:.1%}" if isinstance(v, float) else str(v)

metrics_files = sorted(stage5_dir.glob("metrics_*.json")) if stage5_dir.exists() else []
if metrics_files:
    print("=" * 65)
    print("EVALUATION RESULTS")
    print("=" * 65)
    for mf in metrics_files:
        m = json.loads(mf.read_text())
        m = m.get("metrics", m)
        cam = mf.stem.replace("metrics_", "")
        idf1 = _pct(m.get("IDF1", m.get("idf1", "?")))
        mota = _pct(m.get("MOTA", m.get("mota", "?")))
        hota = _pct(m.get("HOTA", m.get("hota", "?")))
        idsw = m.get("ID_Sw", m.get("id_switches", "?"))
        print(f"  {cam:12s}  IDF1={idf1}  MOTA={mota}  HOTA={hota}  IDsw={idsw}")

for fname in ["summary.json", "evaluation_report.json"]:
    sf = stage5_dir / fname
    if sf.exists():
        s = json.loads(sf.read_text())
        print("-" * 65 + "\n  GLOBAL:")
        for k in ["IDF1","MOTA","HOTA","ID_Sw","idf1","mota","hota","id_switches"]:
            v = s.get(k)
            if v is not None: print(f"    {k}: {_pct(v)}")
        break

if not metrics_files:
    print("No metrics files found -- check stage5 output dir:", stage5_dir)

## 6. Parameter Scan (optional)

Sweep multiple values of a Stage 4 param without touching 10a/10b.

In [ ]:
# Uncomment ONE scan block at a time:

# for aqe_k in [3, 5, 7, 10, 15]:
#     print(f"\n=== aqe_k={aqe_k} ===")
#     subprocess.run([
#         sys.executable, "scripts/scan_stage4_params.py",
#         "--run", RUN_NAME, "--scan", "aqe_k",
#         "--output-dir", str(DATA_OUT),
#     ], cwd=str(PROJECT))

# for res in [0.5, 0.7, 0.9, 1.1, 1.3]:
#     print(f"\n=== louvain_res={res} ===")
#     subprocess.run([
#         sys.executable, "scripts/scan_stage4_params.py",
#         "--run", RUN_NAME, "--scan", "louvain_res",
#         "--output-dir", str(DATA_OUT),
#     ], cwd=str(PROJECT))

print("Uncomment a scan loop above and re-run this cell.")